# Libraries

In [1]:
!pip -q install "lm_eval[hf]"
!pip -q install compressed-tensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 76.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.6 MB/s eta 0:00:00


In [2]:
# 2. Libraries
import random
import numpy as np
import torch
import os
import gc

import lm_eval
import subprocess

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import shutil
from IPython.display import FileLink

# Functions

In [3]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Global Variables

In [4]:
model_name = "EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit"
SEED = 42
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Seed for reproducibility

In [5]:
set_reproducibility(SEED)

Global seed set to 42


# Login to huggingface

In [6]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Do Evaluation

**Configurations**

In [7]:
tasks_config = {
    "gsm8k": {
        "task_name": "gsm8k_cot",
        "fewshot": 2   # 🔻 reduced from 8
    },
    "mmlu": {
        "task_name": "mmlu",
        "fewshot": 2   # 🔻 reduced from 5
    },
    "arc_challenge": {
        "task_name": "arc_challenge",
        "fewshot": 0   # keep
    },
    "wikitext": {
        "task_name": "wikitext",
        "fewshot": 0   # keep
    }
}

In [8]:
for task, cfg in tasks_config.items():
    print(f"Starting evaluation for: {task}")

    command = [
        "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={model_name},max_length=2048,dtype=float16,parallelize=True",
        "--tasks", cfg["task_name"],
        "--batch_size", str(2),   
        "--verbosity", "WARNING",
        "--seed", str(SEED),
        "--limit", str(100),
        "--num_fewshot", str(cfg["fewshot"]),
        "--output_path", f"evaluation_results/{task}"
    ]

    subprocess.run(command, check=True)
    
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Finished {task}\n")

Starting evaluation for: gsm8k


2026-03-26:14:48:14 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-26:14:48:20 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-03-26:14:48:20 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-03-26 14:48:35.129978: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774536515.319612     138 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774536515.373962     138 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plu

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 2
|  Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value|   |Stderr|
|---------|------:|----------------|-----:|-----------|---|----:|---|-----:|
|gsm8k_cot|      3|flexible-extract|     2|exact_match|↑  | 0.55|±  |0.0500|
|         |       |strict-match    |     2|exact_match|↑  | 0.48|±  |0.0502|

Finished gsm8k

Starting evaluation for: mmlu


2026-03-26:15:25:54 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-26:15:25:59 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
2026-03-26:15:25:59 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-03-26 15:26:05.928754: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774538765.949210     266 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774538765.954484     266 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin c

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 2
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.5858|±  |0.0062|
| - humanities                          |      2|none  |      |acc   |↑  |0.6338|±  |0.0130|
|  - formal_logic                       |      1|none  |     2|acc   |↑  |0.4400|±  |0.0499|
|  - high_school_european_history       |      1|none  |     2|acc   |↑  |0.7300|±  |0.0446|
|  - high_school_us_history             |      1|none  |     2|acc   |↑  |0.7500|±  |0.0435|
|  - high_school_world_history          |      1|none  |     2|acc   |↑  |0.7400|±  |0.0441|
|  - international_law                  |      1|none  |     2|

2026-03-26:16:00:16 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-26:16:00:21 INFO     [_cli.run:376] Selected Tasks: ['arc_challenge']
2026-03-26:16:00:21 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-03-26 16:00:29.755966: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774540829.777392    1018 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774540829.782723    1018 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 0, batch_size: 2
|    Tasks    |Version|Filter|n-shot| Metric |   |Value|   |Stderr|
|-------------|------:|------|-----:|--------|---|----:|---|-----:|
|arc_challenge|      1|none  |     0|acc     |↑  | 0.45|±  |0.0500|
|             |       |none  |     0|acc_norm|↑  | 0.43|±  |0.0498|

Finished arc_challenge

Starting evaluation for: wikitext


2026-03-26:16:02:08 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-26:16:02:13 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
2026-03-26:16:02:13 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-03-26 16:02:20.498430: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774540940.519935    1096 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774540940.525255    1096 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plug

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 0, batch_size: 2
| Tasks  |Version|Filter|n-shot|    Metric     |   | Value |   |Stderr|
|--------|------:|------|-----:|---------------|---|------:|---|------|
|wikitext|      2|none  |     0|bits_per_byte  |↓  | 0.7177|±  |   N/A|
|        |       |none  |     0|byte_perplexity|↓  | 1.6445|±  |   N/A|
|        |       |none  |     0|word_perplexity|↓  |14.2980|±  |   N/A|

Finished wikitext



# Save reports

In [9]:
zip_path = shutil.make_archive(f"evaluation_results", 'zip', f"evaluation_results")
FileLink(zip_path)

/kaggle/working/evaluation_results.zip